# A secure FastAPI service, tested in-process

**Module 16 — Secure AI Services and APIs**

A model endpoint is an interface with a contract, an identity, a budget and a log. This notebook builds that
endpoint with FastAPI, then drives it through `httpx` in-process — no server, no network — and checks the status
code for every failure mode: missing key, oversized body, invalid payload, exhausted quota, and a valid call.

## Learning objectives

1. Separate the request schema from the internal model call.
2. Authenticate with an API key and return a uniform error contract.
3. Apply a per-key quota and a body-size limit at the edge.
4. Test the service in-process with realistic status codes.
5. Verify that error responses leak neither internals nor stack traces.

## Background

The endpoint's job is to be boring under abuse:

| Condition | Status | Body |
|---|---|---|
| no or wrong API key | 401 | generic message |
| payload fails schema | 422 | field errors only |
| body too large | 413 | generic message |
| quota exhausted | 429 | retry hint |
| model unavailable | 503 | generic message |

`500` with a traceback is a finding: it tells an attacker the framework, the file paths and often the query.
The handler must convert internal failures into a fixed contract.

Keys are read from the environment in production. Here a mock key is defined in the notebook purely so the test
is self-contained; no real credential is used anywhere.

## Dependencies and installation

```bash
python -m pip install "fastapi>=0.115" "httpx>=0.27" "pydantic>=2.7" "pandas>=2.2"
```

CPU, seconds, no network. `httpx.ASGITransport` calls the app object directly, so nothing binds a port.

In [1]:
import httpx
import pandas as pd
from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field

pd.set_option("display.width", 150)
print("ready")

ready


**Expected output** — `ready`.

### Step 1 — The app: schema, auth, quota, size limit

Read the order of operations. Size and auth are checked before the model is touched; the model call is wrapped so
that any internal error becomes a fixed 503.

In [2]:
app = FastAPI(title="Toy classifier service")

VALID_KEYS = {"test-key-alpha": {"quota": 3}}
MAX_BODY_BYTES = 4096


class PredictRequest(BaseModel):
    text: str = Field(min_length=1, max_length=500)
    top_k: int = Field(default=1, ge=1, le=3)


def score(text: str, top_k: int):
    # Stand-in for a model. Deterministic and cheap.
    labels = ["billing", "technical", "other"]
    weights = [len(text) % 3, (len(text) // 3) % 3, (len(text) // 9) % 3]
    total = sum(weights) or 1
    ranked = sorted(zip(labels, [w / total for w in weights]), key=lambda kv: -kv[1])
    return [{"label": label, "score": round(p, 3)} for label, p in ranked[:top_k]]


@app.middleware("http")
async def limit_body(request: Request, call_next):
    body = await request.body()
    if len(body) > MAX_BODY_BYTES:
        return JSONResponse({"detail": "request too large"}, status_code=413)
    request._body = body
    return await call_next(request)


@app.post("/predict")
async def predict(payload: PredictRequest, x_api_key: str = Header(default="")):
    record = VALID_KEYS.get(x_api_key)
    if record is None:
        raise HTTPException(status_code=401, detail="invalid credentials")
    if record["quota"] <= 0:
        raise HTTPException(status_code=429, detail="quota exhausted", headers={"Retry-After": "60"})
    record["quota"] -= 1
    try:
        return {"predictions": score(payload.text, payload.top_k)}
    except Exception:
        # Never surface the internal exception to the caller.
        raise HTTPException(status_code=503, detail="service temporarily unavailable")


print("routes:", [r.path for r in app.routes if getattr(r, "path", "").startswith("/predict")])

routes: ['/predict']


**Expected output** — `routes: ['/predict']`. One endpoint, four controls in front of it, and a quota that lives
in memory (fine for a demonstration, wrong for production — see the exercises).

### Step 2 — Drive the service in-process

A helper that posts a payload and returns the status and a short body. Using ASGI transport means the test is
fast and deterministic and needs no port.

In [3]:
async def call(payload, key="test-key-alpha", raw=None):
    transport = httpx.ASGITransport(app=app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
        headers = {"x-api-key": key} if key else {}
        if raw is not None:
            resp = await client.post("/predict", content=raw, headers=headers)
        else:
            resp = await client.post("/predict", json=payload, headers=headers)
    body = resp.text[:120].replace("\n", " ")
    return resp.status_code, body


status, body = await call({"text": "App crashes on launch", "top_k": 2})
print(status, body)

200 {"predictions":[{"label":"other","score":0.667},{"label":"technical","score":0.333}]}


**Expected output** — `200` and a JSON body with two predictions. Quota is now 2 of 3.

### Step 3 — The failure matrix

Every row is a request the service must refuse, plus one it must accept. The expected status is declared before
the call.

In [4]:
tests = [
    ("valid request", {"text": "App crashes on launch", "top_k": 2}, "test-key-alpha", 200),
    ("missing key", {"text": "hello"}, None, 401),
    ("wrong key", {"text": "hello"}, "not-a-key", 401),
    ("empty text", {"text": ""}, "test-key-alpha", 422),
    ("top_k too large", {"text": "hello", "top_k": 9}, "test-key-alpha", 422),
    ("wrong type", {"text": 42}, "test-key-alpha", 422),
    ("oversized body", None, "test-key-alpha", 413),
]

rows = []
for name, payload, key, expected in tests:
    if name == "oversized body":
        status, body = await call(None, key=key, raw=b"x" * 5000)
    else:
        status, body = await call(payload, key=key)
    rows.append({"case": name, "expected": expected, "actual": status, "pass": status == expected,
                 "body": body})
matrix = pd.DataFrame(rows)
matrix[["case", "expected", "actual", "pass"]]

,case,expected,actual,pass
0,valid request,200,200,True
1,missing key,401,401,True
2,wrong key,401,401,True
3,empty text,422,422,True
4,top_k too large,422,422,True
5,wrong type,422,422,True
6,oversized body,413,413,True


**Expected output** — all seven rows `pass: True`. Check the `body` column too: 401 and 413 return a short
generic message, and 422 returns field-level detail without internals.

### Step 4 — Exhaust the quota, and read the retry hint

The limiter must be observable: a client that cannot tell "slow down" from "you are broken" will retry
immediately and make it worse.

In [5]:
VALID_KEYS["test-key-alpha"]["quota"] = 3          # reset so this cell is self-contained
results = []
for i in range(4):
    status, body = await call({"text": f"request {i}"}, key="test-key-alpha")
    results.append((i + 1, status, body))
for n, status, body in results:
    print(f"call {n}: {status} {body}")

status_other, _ = await call({"text": "hello"}, key="not-a-key")
print("still 401 for a bad key:", status_other)
assert results[-1][1] == 429 and status_other == 401

call 1: 200 {"predictions":[{"label":"other","score":1.0}]}
call 2: 200 {"predictions":[{"label":"other","score":1.0}]}
call 3: 200 {"predictions":[{"label":"other","score":1.0}]}
call 4: 429 {"detail":"quota exhausted"}
still 401 for a bad key: 401


**Expected output** — calls 1 to 3 return 200, call 4 returns 429 with a `Retry-After` header, and a bad key
still returns 401 regardless of quota. The assertion encodes both invariants: quota applies per key, and
authentication is decided first.

### Step 5 — Check that errors do not leak

Scan every recorded body for framework names, file paths and tracebacks. This is the check that catches a
well-meaning `raise` inside a handler.

In [6]:
leak_markers = ["Traceback", "site-packages", "File \"", "fastapi", "pydantic", "KeyError"]
leaks = []
for row in matrix.itertuples():
    hits = [m for m in leak_markers if m.lower() in row.body.lower()]
    if hits:
        leaks.append({"case": row.case, "markers": hits})
print("responses checked:", len(matrix), "| leaks found:", len(leaks))
assert not leaks
leaks

responses checked: 7 | leaks found: 0


[]

**Expected output** — no leaks, and an empty table. If a handler raises an unexpected exception and FastAPI's
debug mode is on, this check fails — which is exactly the point of running it before deployment.

## Exercises

1. **Move the quota out of memory.** Replace `VALID_KEYS` quota counters with a store that survives restarts, and
   describe the race when two requests arrive simultaneously.
2. **Add request ids.** Return an `X-Request-Id` and include it in the log line. Why is this the first thing an
   on-call engineer asks for?
3. **Rate limit by key.** Combine the quota with a token bucket from the previous notebook and decide which limit
   produces which status code.
4. **Write the contract test.** Turn the failure matrix into a test that runs in CI and fails the build when a
   status changes unexpectedly.

## Reflection questions

1. Why must authentication be decided before quota, and what breaks if it is not?
2. A 503 hides the internal error from the caller. Who should see it, and how?
3. Which control here is defeated by an attacker with many API keys, and what replaces it?

## References

1. FastAPI documentation — dependencies, middleware and exception handling.
2. OWASP, *API Security Top 10*, 2023 — API1, API4, API6.
3. OWASP, *ASVS* 4.0.3, V7 (error handling and logging).
4. NIST, *SP 800-63B* — throttling and abuse resistance.